In [110]:
import requests
from bs4 import BeautifulSoup
from time import sleep
import pandas as pd

In [37]:
headers = headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}

In [152]:
def get_url():
    for page in range(1, 12):
        sleep(3)
        main_url = f'https://www.goszakup.gov.kz/ru/registry/rqc?count_record=50&page={page}'
        response = requests.get(main_url, headers=headers)
        soup = BeautifulSoup(response.text, 'lxml')
        data = soup.find('tbody').find_all('tr')
        
        for i in data:
            url = i.find('a').get('href')
            yield url

In [159]:
info = []

for url in get_url():
    bin_v = name = fio = iin = address = None
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'lxml')
    rows = soup.find_all('tr')
    
    for row in rows:
        th = row.find('th')
        td = row.find('td')
        if not th or not td:
            continue
        
        field = th.text.strip()
        value = td.text.strip()
        if field == 'БИН участника':
            bin_v = value
        elif field == 'Наименование на рус. языке':
            name = value
        elif field == 'ИИН':
            iin = value
        elif field == 'ФИО':
            fio = value

    for row in rows:
        th = row.find_all('th')
        if len(th) <= 1:
            continue
    
        ths = [h.get_text(strip = True) for h in th]
        
        if 'Полный адрес(рус)' in ths:
            idx = ths.index('Полный адрес(рус)')

            curr_idx = rows.index(row)
            td_row = rows[curr_idx + 1]
            td = td_row.find_all('td')
            tds = [d.get_text(strip = True) for d in td]
            address = tds[idx]
            break

    info.append({
        'Наименование организации': name,
        'БИН организации': bin_v,
        'ФИО руководителя': fio,
        'ИИН руководителя': iin,
        'Полный адрес организации': address
    })

In [160]:
df = pd.DataFrame(info).drop_duplicates()
df

,Наименование организации,БИН организации,ФИО руководителя,ИИН руководителя,Полный адрес организации
0,"Товарищество с ограниченной ответственностью ""...",160540016411,КОЖАНТАЕВ АЙДАР АСКАРОВИЧ,870811351023,"Северо-Казахстанская область, г.Петропавловск,..."
1,"Товарищество с ограниченной ответственностью""G...",070540008075,ХИЛАЖЕВ АНВАР ОЛЕГОВИЧ,760315301474,"Алматинская область, Талгарский район, г.Талга..."
2,"Товарищество с ограниченной ответственностью ""...",110740014027,ЖАНИБЕКОВ НУРЛАН СЕЙЛХАНОВИЧ,790908300861,"г.Алматы, Медеуский район, мкр. Алатау, Ибраги..."
3,"Товарищество с ограниченной ответственностью ""...",130740024484,ЧИРЬЕВ ЕВГЕНИЙ ВАДИМОВИЧ,810515350203,"г.Астана, Жилой массив КОКТАЛ , Улица БОЛАШАК, 34"
4,"Товарищество с ограниченной ответственностью ""...",000440001170,ИВАНОВА ОЛЬГА НИКОЛАЕВНА,810210401550,"Восточно-Казахстанская область, г.Усть-Каменог..."
...,...,...,...,...,...
531,AS & ER GRADE товарищества с ограниченной отве...,120940007043,АУЕЛБЕКОВ ЕРКИН ЕРЖАНОВИЧ,871201300065,"Туркестанская область, Ордабасынский район, с...."
532,"ИП ""Батыралиев Н.М.""",,БАТЫРАЛИЕВ НУРСУЛТАН МАРАТАЛИЕВИЧ,930706302177,"г.Алматы, Алатауский район, мкр. Дархан ул. Ум..."
533,"Учреждение ""Атырауское учебно-производственное...",921040000599,СЕЙДОВ АЛИ КАДИРОВИЧ,550501300423,"Атырауская область, г.Атырау, ФРОЛОВ, 7"
534,"Товарищество с ограниченной ответственностью ""...",180840031777,СУЛЕЙМАНОВ АЛМАТ СЕРИКОВИЧ,850304301176,"область Абай, г.Семей, Гастелло, 1"


In [161]:
df.to_excel('goszakup_results.xlsx', index=False)